# core

> Workflow-level route handlers for status, reset, and source retrieval

In [ ]:
#| default_exp routes.core

In [ ]:
#| export
from pathlib import Path

from fasthtml.common import Div
from starlette.responses import FileResponse, Response

from cjm_fasthtml_card_stack.components.controls import render_width_slider
from cjm_fasthtml_card_stack.core.constants import DEFAULT_VISIBLE_COUNT, DEFAULT_CARD_WIDTH

from cjm_fasthtml_interactions.core.state_store import get_session_id

from cjm_fasthtml_workflow_transcript_decomp.core.html_ids import StructureDecompHtmlIds
from cjm_fasthtml_workflow_transcript_decomp.core.models import WorkingSegment, VADChunk
from cjm_fasthtml_workflow_transcript_decomp.routes.models import DecompUrls, AlignmentUrls
from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

# Decomposition renderers
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.step_renderer import (
    render_toolbar as render_decomp_toolbar,
    render_decomp_footer_content,
)
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.card_stack_config import (
    DECOMP_CS_CONFIG, DECOMP_CS_IDS,
)

# Alignment renderers
from cjm_fasthtml_workflow_transcript_decomp.components.step_alignment.step_renderer import (
    render_align_toolbar, render_align_footer_content,
)
from cjm_fasthtml_workflow_transcript_decomp.components.step_alignment.card_stack_config import (
    ALIGN_CS_CONFIG, ALIGN_CS_IDS,
)

# Keyboard config
from cjm_fasthtml_workflow_transcript_decomp.components.keyboard_config import (
    build_combined_kb_system, render_keyboard_hints_collapsible,
)

# Debug flags
DEBUG_AUDIO = False
DEBUG_SWITCH_CHROME = False

## Core Route Handlers

These handlers manage workflow status and lifecycle.

In [ ]:
#| export
def _handle_current_status(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess  # FastHTML session object
):  # Appropriate UI component based on current state
    """Return current workflow status - determines what to show."""
    session_id = get_session_id(sess)
    
    # Check for in-progress workflow (via server-side state store)
    workflow_state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    current_step = workflow.state_store.get_current_step(workflow.config.workflow_id, session_id)
    
    # Restore external paths from state to SourceService (fixes persistence across restarts)
    step_states = workflow_state.get("step_states", {})
    selection_state = step_states.get("selection", {})
    external_db_paths = selection_state.get("external_db_paths", [])
    if external_db_paths:
        workflow.source_service.set_external_paths(external_db_paths)
    
    # If there's existing state, resume the workflow
    if current_step or workflow_state:
        return workflow._stepflow_router.start(request, sess)
    
    # Start fresh
    return workflow._stepflow_router.start(request, sess)

In [ ]:
#| export
def _handle_reset(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess  # FastHTML session object
):  # StepFlow start view
    """Reset workflow and return to start."""
    session_id = get_session_id(sess)
    workflow.state_store.clear_state(workflow.config.workflow_id, session_id)
    return workflow._stepflow_router.start(request, sess)

In [ ]:
#| export
def _handle_get_sources(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    provider_id: str = None,  # Optional plugin name filter
    limit: int = 50  # Maximum number of results
):  # JSON response with transcription sources
    """Get available transcription sources."""
    transcriptions = workflow.source_service.query_transcriptions(
        provider_id=provider_id,
        limit=limit
    )
    return {"transcriptions": transcriptions}

## Combined Step Handlers

Handlers for the combined Phase 2 step: chrome switching and audio serving.

In [ ]:
#| export
async def _handle_switch_chrome(
    workflow:StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    decomp_urls:DecompUrls,  # URL bundle for decomposition routes
    align_urls:AlignmentUrls,  # URL bundle for alignment routes
) -> tuple:  # OOB swaps for shared chrome containers
    """Switch shared chrome content based on active column."""
    form = await request.form()
    active_column = form.get("active_column", "decomp")

    if DEBUG_SWITCH_CHROME:
        print(f"[SWITCH_CHROME] active_column: {active_column}")

    session_id = get_session_id(sess)
    state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    step_states = state.get("step_states", {})
    decomp_state = step_states.get("decomposition", {})
    align_state = step_states.get("alignment", {})

    # Build combined KB manager for hints
    kb_manager, _ = build_combined_kb_system(decomp_urls, align_urls)

    if active_column == "decomp":
        # Decomp chrome
        segments = [WorkingSegment.from_dict(s) for s in decomp_state.get("segments", [])]
        history = decomp_state.get("history", [])
        focused_index = decomp_state.get("focused_index", 0)
        visible_count = decomp_state.get("visible_count", DEFAULT_VISIBLE_COUNT)
        card_width = decomp_state.get("card_width", DEFAULT_CARD_WIDTH)

        hints_content = render_keyboard_hints_collapsible(kb_manager, include_zone_switch=True)
        toolbar_content = render_decomp_toolbar(
            reset_url=decomp_urls.reset,
            ai_split_url=decomp_urls.ai_split,
            undo_url=decomp_urls.undo,
            can_undo=(len(history) > 0),
            visible_count=visible_count,
        )
        controls_content = render_width_slider(DECOMP_CS_CONFIG, DECOMP_CS_IDS, card_width=card_width)
        footer_content = render_decomp_footer_content(segments, focused_index)
    else:
        # Alignment chrome
        chunks = [VADChunk.from_dict(c) for c in align_state.get("vad_chunks", [])]
        history = align_state.get("history", [])
        focused_index = align_state.get("focused_chunk_index", 0)
        visible_count = align_state.get("visible_count", 5)
        card_width = align_state.get("card_width", 40)

        hints_content = render_keyboard_hints_collapsible(kb_manager, include_zone_switch=True)
        toolbar_content = render_align_toolbar(
            auto_align_url=align_urls.auto_align,
            clear_url=align_urls.clear_assignments,
            undo_url=align_urls.undo,
            can_undo=(len(history) > 0),
            visible_count=visible_count,
        )
        controls_content = render_width_slider(ALIGN_CS_CONFIG, ALIGN_CS_IDS, card_width=card_width)
        footer_content = render_align_footer_content(chunks, focused_index)

    if DEBUG_SWITCH_CHROME:
        print(f"[SWITCH_CHROME] returning OOB swaps for {active_column}")

    # Return OOB swaps
    hints_oob = Div(
        hints_content,
        id=StructureDecompHtmlIds.SHARED_HINTS,
        hx_swap_oob="innerHTML"
    )
    toolbar_oob = Div(
        toolbar_content,
        id=StructureDecompHtmlIds.SHARED_TOOLBAR,
        hx_swap_oob="innerHTML"
    )
    controls_oob = Div(
        controls_content,
        id=StructureDecompHtmlIds.SHARED_CONTROLS,
        hx_swap_oob="innerHTML"
    )
    footer_oob = Div(
        footer_content,
        id=StructureDecompHtmlIds.SHARED_FOOTER,
        hx_swap_oob="innerHTML"
    )

    return (hints_oob, toolbar_oob, controls_oob, footer_oob)

In [ ]:
#| export
def _handle_audio_src(
    workflow:StructureDecompWorkflow,  # The workflow instance
    sess,  # FastHTML session object
):  # Audio file response or 404
    """Serve the audio file for the current alignment session."""
    session_id = get_session_id(sess)

    if DEBUG_AUDIO:
        print(f"[AUDIO_SRC] Serving audio for session: {session_id}")

    state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    align_state = state.get("step_states", {}).get("alignment", {})
    media_path = align_state.get("media_path")

    if DEBUG_AUDIO:
        print(f"[AUDIO_SRC] media_path from state: {media_path}")
        if media_path:
            print(f"[AUDIO_SRC] file exists: {Path(media_path).is_file()}")

    if not media_path or not Path(media_path).is_file():
        if DEBUG_AUDIO:
            print(f"[AUDIO_SRC] Returning 404 - media_path missing or file not found")
        return Response(status_code=404)

    if DEBUG_AUDIO:
        print(f"[AUDIO_SRC] Returning FileResponse for: {media_path}")

    return FileResponse(str(media_path), media_type="audio/mpeg")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()